In [39]:
#from google.colab import drive
#drive.mount('/content/drive')
#!pip freeze > requirements.txt


In [26]:
import shutil
import pandas as pd
import random

import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging

import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity



import os
import glob
import json
import numpy as np
import tensorflow as tf


In [2]:
import sys
utils_path = 'C:\\Users\\admin\\0.Master_Thesis\\'
sys.path.append(utils_path)

fixed_path = f'{utils_path}CellCNN'
#cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'
if fixed_path not in sys.path:
    sys.path.append(fixed_path)

In [3]:
# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample as downsample


from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation, splitting
from Dataset_utils import show_hyperparameters, phenotype_prediction, default_serializer, remove_from_cache, min_length

In [4]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache
remove_from_cache(decache_files)
# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters, min_length

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


In [ ]:
#FIX NEXT FUNCTION TO IMPORT CORRECTLY THE DATASETS WITHOUT GHENT CODE

# Import data and data transformation


In [5]:
%%time

# Trova tutti i file con estensione specifica in una cartella
data_folder = f'{utils_path}B-ALL_Datasets'
extension = '*.csv'  # cambia con l'estensione che ti serve

files_list = glob.glob(os.path.join(data_folder, extension))

ALL_DATASETS = []
counter = 0
multiple_donations = {}
no_id = []
for file_path in files_list:
    #import the dataset
    dataset = pd.read_csv(file_path, sep = ';', decimal = ',')
    ALL_DATASETS.append(dataset) # list of all datasets

    # divide the datasets by donors
    multiple_donations = patient_code_extraction(file_path, counter, multiple_donations)
    
    print(f"Elaborando file {counter}: {file_path}") # information about the process
    
    counter += 1 

# Fix no_id datasets
last_identifier = 0
for element in multiple_donations.keys():
    if element.isdigit():
        if int(element) > int(last_identifier):
            last_identifier = int(element)
print(last_identifier)
for dataset in multiple_donations['no_id']:
    last_identifier += 1
    multiple_donations[str(last_identifier)] = [dataset]
multiple_donations.pop('no_id')

Elaborando file 0: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220204-2900.csv
Elaborando file 1: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220208-3665.csv
Elaborando file 2: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_220216-3546.csv
Elaborando file 3: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D15_1.csv
Elaborando file 4: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D29_1.csv
Elaborando file 5: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE11_D71_1.csv
Elaborando file 6: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D15_2.csv
Elaborando file 7: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE12_D29_1.csv
Elaborando file 8: C:\Users\admin\0.Master_Thesis\B-ALL_Datasets\B-ALL_expression_matrix_B-ALL_GHE1_D15_2.csv
Elaborando file 9: C

[0, 1, 2]

In [ ]:
# Show patients donations
print(multiple_donations)

In [6]:
healthy_donors, blast_donors, mixed_donors = donor_division(multiple_donations, ALL_DATASETS)
#print(first_subset)
#testing_set_idx = list(set(range(len(files_list))) - set(training_set_idx) - set(val_set_idx))
#print(training_set_idx, val_set_idx, testing_set_idx)
print(healthy_donors, blast_donors, mixed_donors)

{'11': [1, 1, 0], '12': [1, 1], '1': [1, 0], '2': [1, 1], '3': [1, 0], '4': [1, 1, 0], '6': [1, 1], '7': [1, 1, 0], '8': [1, 1], '9': [1, 1], '13': [0], '14': [0], '15': [0]}
['12', '2', '6', '8', '9'] ['13', '14', '15'] ['11', '1', '3', '4', '7']


In [38]:
random.seed(42)

train_donors = []
val_donors = []
test_donors = []

healthy_donors_idx = random.sample(list(range(len(healthy_donors))), len(healthy_donors))
blast_donors_idx = random.sample(list(range(len(blast_donors))), len(blast_donors))
mixed_donors_idx = random.sample(list(range(len(mixed_donors))), len(mixed_donors))
print(healthy_donors_idx, blast_donors_idx, mixed_donors_idx)

#train_donors_idx, val_donors_idx, test_donors_idx = splitting(healthy_donors, blast_donors, mixed_donors, healthy_donors_idx, blast_donors_idx, mixed_donors_idx)
#print(train_donors_idx, val_donors_idx, test_donors_idx)

[0, 4, 2, 1, 3] [0, 2, 1] [4, 0, 2, 1, 3]


In [ ]:
# extract donors datasets from the entire natch od datasets and place them into correct dataset
raw_train_datasets = dataset_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
raw_val_datasets = dataset_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
raw_test_datasets = dataset_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

#print(raw_train_datasets, raw_val_datasets, raw_test_datasets)

In [ ]:
%%time
n_sub = 3
raw_train_sampled_subsets = subsample_generation(raw_train_datasets, n_sub)
raw_val_sampled_subsets = subsample_generation(raw_val_datasets, n_sub)
raw_test_sampled_subsets = subsample_generation(raw_test_datasets, n_sub)
print(len(raw_train_sampled_subsets))
print(len(raw_val_sampled_subsets))
print(len(raw_test_sampled_subsets))


In [ ]:
train_datasets, train_y = retrieve_labels(raw_train_sampled_subsets, log = False)
val_datasets, val_y = retrieve_labels(raw_val_sampled_subsets, log = False)
test_datasets, test_y = retrieve_labels(raw_test_sampled_subsets, log = False)

In [ ]:
print(len(train_datasets), train_y)
print(len(val_datasets), val_y)
print(len(test_datasets), test_y)

In [ ]:
cv_train_datasets = train_datasets + val_datasets
cv_train_y = train_y + val_y

In [ ]:
print(len(cv_train_datasets), cv_train_y)

In [ ]:
test_datasets_no_labels = []
for df in test_datasets:
    test_datasets_no_labels.append(df.drop(columns = ['IsBlast']).values)


In [ ]:
# extract the length of the smallest dataset in the training dataset

min_l = min_length(train_datasets)
print(min_l)

In [14]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache
remove_from_cache(decache_files)
# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache

cellCnn.model_new_unfixed rimosso dalla cache
cellCnn.utils_new_unfixed rimosso dalla cache
Dataset_Elaboration_Class rimosso dalla cache
Dataset_utils rimosso dalla cache
cellCnn.downsample_new_unfixed rimosso dalla cache


In [ ]:
%%time
model = CellCnn(
    ncell=100,            #200                        # Number of cells per multi-cell input (sampled from the 'patient' datasets)
    nsubset=int(1),                                    # Total number of multi-cell inputs that will be generated per class (or sample and class)
    per_sample = True,                                # For each sample, nsubset samples of ncell are performed
    nfilter_choice= [3,5,7,9], #list(range(3,21)),                 # Range of possible number of filters
    maxpool_percentages=[0.01, 1., 5., 20., 100.],    # list of k-percentage max_pooling
    learning_rate=[0.01],                               # Learning rate
    max_epochs=1, #50
    patience=5,                                       # Early stopping patience
    nrun=1,  #15                                     # Number of neural network configurations to try (for Hyperparameter optimization)
    regression=False,
    scale=True,                                       # Z-score normalization
    verbose=1
)

outdir = '/content/cellcnn_results'  # Results Directory
model.fit(
        train_samples=train_datasets,
        train_phenotypes=np.array(train_y),
        outdir=outdir,
        valid_samples = val_datasets,
        valid_phenotypes=np.array(val_y),
        generate_valid_set = False
    )


In [ ]:
%%time
test_pred = model.predict((test_datasets_no_labels))


In [ ]:
accuracy_list = []
prediction = [test_pred]
for pred in prediction:
    
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

In [8]:

def CSNN_restructured(healthy_donors, blast_donors, mixed_donors, n_cell = 100, n_sub = 3, cv = False, seed = 42, 
                      max_epochs=1, #50
                        patience=5,                                       # Early stopping patience
                        nrun=1):  #15   ):
    train_donors = []
    val_donors = []
    test_donors = []
    
    print(f'Precess starts. Dividing donors...')
    # sammple indexed for donor division
    healthy_donors_idx = random.sample(list(range(len(healthy_donors))), len(healthy_donors))
    blast_donors_idx = random.sample(list(range(len(blast_donors))), len(blast_donors))
    mixed_donors_idx = random.sample(list(range(len(mixed_donors))), len(mixed_donors))
    #print(healthy_donors_idx, blast_donors_idx, mixed_donors_idx)

    print(f'Seting Train, Validation and Test idx...')
    # just divide accoding to the sampled indexes
    train_donors_idx, val_donors_idx, test_donors_idx = splitting(healthy_donors, blast_donors, mixed_donors, healthy_donors_idx, blast_donors_idx, mixed_donors_idx)

    print(f'Extracting data...')
    
    # extract donors datasets from the entire natch od datasets and place them into correct dataset
    raw_train_datasets = dataset_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
    raw_val_datasets = dataset_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
    raw_test_datasets = dataset_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

    
    print(f'Generating Subsamples...')
    # creates subsets for each sample
    raw_train_sampled_subsets = subsample_generation(raw_train_datasets, n_sub, seed = seed)
    raw_val_sampled_subsets = subsample_generation(raw_val_datasets, n_sub, seed = seed)
    raw_test_sampled_subsets = subsample_generation(raw_test_datasets, n_sub, seed = seed)

    
    # retrieve labels from each subset
    train_datasets, train_y = retrieve_labels(raw_train_sampled_subsets, log = False)
    val_datasets, val_y = retrieve_labels(raw_val_sampled_subsets, log = False)
    test_datasets, test_y = retrieve_labels(raw_test_sampled_subsets, log = False)

        # remove cells labels for training
    test_datasets_no_labels = []
    for df in test_datasets:
        test_datasets_no_labels.append(df.drop(columns = ['IsBlast']).values)

    
    # recreate training dataset for cross validation training
    if cv: 
        cv_train_datasets = train_datasets + val_datasets
        cv_train_y = train_y + val_y

    
    # use n_cell equal to the minimum dimension over all datasets (subdatasets)
    if n_cell == 'min':
        n_cell = min_length(train_dataset)
        print(f'n_cell = {min_l}')
    
    model = CellCnn(
        ncell=n_cell,            #200                        # Number of cells per multi-cell input (sampled from the 'patient' datasets)
        nsubset=int(1),                                    # Total number of multi-cell inputs that will be generated per class (or sample and class)
        per_sample = True,                                # For each sample, nsubset samples of ncell are performed
        nfilter_choice= [3,5,7,9], #list(range(3,21)),                 # Range of possible number of filters
        maxpool_percentages=[0.01, 1., 5., 20., 100.],    # list of k-percentage max_pooling
        learning_rate=[0.01], #, 0.1, 1, 10, 50, 100],                               # Learning rate
        max_epochs=50, #50
        patience=5,                                       # Early stopping patience
        nrun=15,  #15                                     # Number of neural network configurations to try (for Hyperparameter optimization)
        regression=False,
        scale=True,                                       # Z-score normalization
        verbose=1
    )

    print(f'Model defined...')
    
    outdir = f'/content/cellcnn_results'  # Results Directory

    print(f'Fitting started...')
    model.fit(
            train_samples=train_datasets,
            train_phenotypes=np.array(train_y),
            outdir=outdir,
            valid_samples = val_datasets,
            valid_phenotypes=np.array(val_y),
            generate_valid_set = False
        )

    print(f'Prediction started...')
    test_pred = model.predict((test_datasets_no_labels))
    print(f'Done')
    return test_pred, model.results #, model.model_sorted_idx)

In [10]:
%%time
n_sub = 3
seed = 42
n_cells = 100000
cv = False
trials = 10

predictions_list = []
results_list = []
#best_indices_list = []
#seed_list = random.sample(list(range(10000)), 10)
seed_list = [7359, 9654, 4557, 106, 2615, 6924, 5574, 4552, 2547, 3527]
#print(seed_list)
for i in range(trials):
    print(f'Trial {i+1} started')
    seed = seed_list[i]
    test_pred, model_param = CSNN_restructured(healthy_donors, blast_donors, mixed_donors, n_cells, n_sub = n_sub, cv = False, seed = seed)
    predictions_list.append(test_pred)
    
    #results = model_param[0]
    #best_indices = model_param[1]
    
    results_list.append(model_param)
    #best_indices_list.append(best_indices)
    
    print(f'Trial {i+1} Done!\n')


Trial 1 started
Precess starts. Dividing donors...
Seting Train, Validation and Test idx...
Extracting data...
[22, 23, 17, 18, 0, 19, 20, 21, 3, 4, 5]
[6, 7, 2, 14, 15, 16]
[10, 11, 24, 25, 1, 8, 9, 12, 13]
Generating Subsamples...
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (33,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (33, 100000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (33, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (18, 2)

# ============================================= #
Run: 0

X_tr shape: (33, 100000, 11)
y_tr shape: (33, 2)
X_v shape: (18, 100000, 11)
y_v shape: (18, 2)
Unique values in y_tr: [0. 1.]


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\tensorflow\python\data\ops\structured_function.py:265: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


Epoch 1/50
1/1 [==============================] - 2s 2s/step - loss: 0.6877 - val_loss: 0.6730
Epoch 2/50
1/1 [==============================] - 1s 978ms/step - loss: 0.6625 - val_loss: 0.6591
Epoch 3/50
1/1 [==============================] - 1s 942ms/step - loss: 0.6308 - val_loss: 0.6503
Epoch 4/50
1/1 [==============================] - 1s 1s/step - loss: 0.6140 - val_loss: 0.6486
Epoch 5/50
1/1 [==============================] - 2s 2s/step - loss: 0.5290 - val_loss: 0.6562
Epoch 6/50
1/1 [==============================] - 1s 1s/step - loss: 0.6144 - val_loss: 0.6652
Epoch 7/50
1/1 [==============================] - 1s 972ms/step - loss: 0.5986 - val_loss: 0.6700
Epoch 8/50
1/1 [==============================] - 1s 1s/step - loss: 0.5759 - val_loss: 0.6722
Epoch 9/50
1/1 [==============================] - 0s 260ms/step - loss: 0.6486

# ============================================= #
Run: 1

X_tr shape: (33, 100000, 11)
y_tr shape: (33, 2)
X_v shape: (18, 100000, 11)
y_v shape: (18, 

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\tensorflow\python\data\ops\structured_function.py:265: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 0s 92ms/step
Done
Trial 1 Done!

Trial 2 started
Precess starts. Dividing donors...
Seting Train, Validation and Test idx...
Extracting data...
[10, 11, 6, 7, 1, 14, 15, 16, 8, 9]
[24, 25, 0, 12, 13]
[22, 23, 17, 18, 2, 3, 4, 5, 19, 20, 21]
Generating Subsamples...
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (30,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (30, 100000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (30, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (15, 2)

# ============================================= #
Run: 0

X_tr shape: (30, 100000, 11)
y_tr shape: (30, 2)
X_v shape: (15, 100000, 11)
y_v shape: (15, 2)
Unique values in y_tr: [0. 1.]


...

2/2 [==============================] - 0s 16ms/step
Done
Trial 9 Done!

Trial 10 started
Precess starts. Dividing donors...
Seting Train, Validation and Test idx...
Extracting data...
[24, 25, 6, 7, 2, 8, 9, 3, 4, 5]
[10, 11, 1, 19, 20, 21]
[17, 18, 22, 23, 0, 14, 15, 16, 12, 13]
Generating Subsamples...
Model defined...
Fitting started...
=== BEFORE CATEGORICAL CONVERSION ===
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (30,)
n_classes: 1
=== AFTER CATEGORICAL CONVERSION ===
x_tr: (30, 100000, 11)
y_tr type: <class 'numpy.ndarray'>
y_tr shape: (30, 2)
y_v type: <class 'numpy.ndarray'>
y_v shape: (18, 2)

# ============================================= #
Run: 0

X_tr shape: (30, 100000, 11)
y_tr shape: (30, 2)
X_v shape: (18, 100000, 11)
y_v shape: (18, 2)
Unique values in y_tr: [0. 1.]


C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\tensorflow\python\data\ops\structured_function.py:265: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


Epoch 1/50
1/1 [==============================] - 1s 1s/step - loss: 0.6935 - val_loss: 0.6849
Epoch 2/50
1/1 [==============================] - 1s 698ms/step - loss: 0.6868 - val_loss: 0.6761
Epoch 3/50
1/1 [==============================] - 1s 625ms/step - loss: 0.6688 - val_loss: 0.6678
Epoch 4/50
1/1 [==============================] - 1s 541ms/step - loss: 0.6719 - val_loss: 0.6613
Epoch 5/50
1/1 [==============================] - 1s 559ms/step - loss: 0.6596 - val_loss: 0.6570
Epoch 6/50
1/1 [==============================] - 1s 547ms/step - loss: 0.6341 - val_loss: 0.6559
Epoch 7/50
1/1 [==============================] - 1s 521ms/step - loss: 0.6404 - val_loss: 0.6578
Epoch 8/50
1/1 [==============================] - 1s 500ms/step - loss: 0.5923 - val_loss: 0.6635
Epoch 9/50
1/1 [==============================] - 1s 500ms/step - loss: 0.6724 - val_loss: 0.6650
Epoch 10/50
1/1 [==============================] - 1s 500ms/step - loss: 0.6359 - val_loss: 0.6643
Epoch 11/50
1/1 [=====

C:\Users\admin\anaconda3\envs\tf_env\lib\site-packages\tensorflow\python\data\ops\structured_function.py:265: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


1/1 [==============================] - 0s 109ms/step
Done
Trial 10 Done!

CPU times: total: 5h 31min 39s
Wall time: 2h 42min 41s


In [13]:
print(results_list)

[{'clustering_result': {'w': array([[-2.06470825e-02, -6.28054440e-02,  3.43592912e-02,
         5.84692657e-02,  3.21817137e-02,  2.68634744e-02,
         5.53240664e-02,  7.71962479e-02,  2.54441164e-02,
        -8.15041736e-02, -7.95161948e-02,  3.74412201e-02,
        -8.14834088e-02,  3.13590094e-02],
       [-1.14416648e-02, -4.84917611e-02,  7.45341331e-02,
        -9.93246958e-03,  7.93054476e-02,  7.97016546e-02,
        -6.24447949e-02,  5.21216579e-02, -5.21083474e-02,
        -1.27083557e-02, -3.12870778e-02,  3.63736339e-02,
        -5.56736588e-02,  5.74343204e-02],
       [ 1.14020929e-02,  2.49977559e-02, -4.67294827e-04,
        -1.00618657e-02, -3.53803821e-02, -1.37918862e-03,
         4.56051715e-03,  1.18401553e-02,  1.58310905e-02,
        -3.45702805e-02,  1.00614708e-02, -3.48204337e-02,
        -1.01537863e-03, -2.89640669e-03],
       [ 2.77327076e-02,  6.47559343e-03, -7.13607483e-03,
        -5.55825010e-02,  1.34544848e-02, -9.57338698e-03,
        -1.11198

In [18]:
needed_results = {}
tot_trials_res = []
par_list = ['config', 'model_sorted_idx']
for res in results_list:
    for key, value in res.items():
        if key in par_list:
            needed_results[key] = value
    tot_trials_res.append(needed_results)

In [19]:

import json
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\'

with open(f'{save_path}results_list.json', "w", encoding="utf-8") as f:
    json.dump( tot_trials_res, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{save_path}predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(predictions_list, f, default=default_serializer, ensure_ascii=False, indent=2)

In [ ]:
decache_files = ['cellCnn.model_new_unfixed', 'cellCnn.utils_new_unfixed', 'Dataset_Elaboration_Class', 'Dataset_utils', 'cellCnn.downsample_new_unfixed']
# Rimuovi il modulo specifico dalla cache
remove_from_cache(decache_files)
# import downloaded modules
#from cellCnn.model import CellCnn
from cellCnn.model_new_unfixed import CellCnn
import cellCnn.plotting as plotting
import cellCnn.utils_new_unfixed as utils
#import cellCnn.utils as utils
import cellCnn.downsample_new_unfixed as downsample



from Dataset_Elaboration_Class import read_data, dataset_split, df_to_array, retrieve_labels
from Dataset_Elaboration_Class import patient_code_extraction, divide_donations,  idx_to_dataset
from Dataset_Elaboration_Class import class_division, train_val_samples, concatenate_hb_datasets, donor_division
from Dataset_Elaboration_Class import dataset_extraction, generate_subsets_length , subsample_generation
from Dataset_utils import extract_hyper, phenotype_prediction, default_serializer, remove_from_cache, show_hyperparameters

In [27]:
for i in range(trials):
    res = results_list[i]
    best_models_i = res['model_sorted_idx']
    #print(len(best_models_i))
    config = res['config']
    #print(config)
    trial_par = extract_hyper(config, best_models_i)[:3]
    #print(trial_par)
    show_hyperparameters(trial_par)
    #print(trial_par)
    
    #for j in best_models_i:
    #    print(config[j])
    #trial_parameters = show_hyperparameters(res)

Model 1
Filters: 9, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 2
Filters: 9, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 3
Filters: 7, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 1
Filters: 3, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 2
Filters: 7, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 3
Filters: 3, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 1
Filters: 5, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 2
Filters: 3, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 3
Filters: 7, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 1
Filters: 5, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 2
Filters: 5, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 3
Filters: 7, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 1
Filters: 3, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 


Model 2
Filt

In [ ]:
'''
{'nfilter': [9, 7, 9], 'learning_rate': [], 'maxpool_percentage': [0.01, 1.0, 5.0]}
[[9, 0, 5.0], [7, 0, 1.0], [9, 0, 0.01]]
Model 1
Filters: 9, Learning Rate: 0, Top-k Percentage Max Pooling: 5.0 


Model 2
Filters: 7, Learning Rate: 0, Top-k Percentage Max Pooling: 1.0 


Model 3
Filters: 9, Learning Rate: 0, Top-k Percentage Max Pooling: 0.01 
'''

In [23]:
accuracy_list = []

for pred in predictions_list:
    
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

NameError: name 'test_y' is not defined

# CV

In [ ]:
    train_donors = []
    val_donors = []
    test_donors = []
    n_sub = 3
    n_cell = 100000
    print(f'Precess starts. Dividing donors...')
    # sammple indexed for donor division
    healthy_donors_idx = random.sample(list(range(len(healthy_donors))), len(healthy_donors))
    blast_donors_idx = random.sample(list(range(len(blast_donors))), len(blast_donors))
    mixed_donors_idx = random.sample(list(range(len(mixed_donors))), len(mixed_donors))
    #print(healthy_donors_idx, blast_donors_idx, mixed_donors_idx)

    print(f'Seting Train, Validation and Test idx...')
    # just divide accoding to the sampled indexes
    train_donors_idx, val_donors_idx, test_donors_idx = splitting(healthy_donors, blast_donors, mixed_donors, healthy_donors_idx, blast_donors_idx, mixed_donors_idx)

    print(f'Extracting data...')
    
    # extract donors datasets from the entire natch od datasets and place them into correct dataset
    raw_train_datasets = dataset_extraction(train_donors_idx, multiple_donations, ALL_DATASETS)
    raw_val_datasets = dataset_extraction(val_donors_idx, multiple_donations, ALL_DATASETS)
    raw_test_datasets = dataset_extraction(test_donors_idx, multiple_donations, ALL_DATASETS)

    
    print(f'Generating Subsamples...')
    # creates subsets for each sample
    raw_train_sampled_subsets = subsample_generation(raw_train_datasets, n_sub, seed = seed)
    raw_val_sampled_subsets = subsample_generation(raw_val_datasets, n_sub, seed = seed)
    raw_test_sampled_subsets = subsample_generation(raw_test_datasets, n_sub, seed = seed)

    
    # retrieve labels from each subset
    train_datasets, train_y = retrieve_labels(raw_train_sampled_subsets, log = False)
    val_datasets, val_y = retrieve_labels(raw_val_sampled_subsets, log = False)
    test_datasets, test_y = retrieve_labels(raw_test_sampled_subsets, log = False)

    # remove cells labels for training
    test_datasets_no_labels = []
    for df in test_datasets:
        test_datasets_no_labels.append(df.drop(columns = ['IsBlast']).values)

    
    # recreate training dataset for cross validation training
    #if cv: 
    CV_train_datasets = train_datasets + val_datasets
    CV_train_y = train_y + val_y

    print(type(CV_train_datasets))
    print(type(CV_train_y))
    print(type(CV_train_datasets[0]))

    #'''
    # use n_cell equal to the minimum dimension over all datasets (subdatasets)
    if n_cell == 'min':
        n_cell = min_length(train_dataset)
        print(f'n_cell = {min_l}')
    
    model = CellCnn(
        ncell=n_cell,            #200                        # Number of cells per multi-cell input (sampled from the 'patient' datasets)
        nsubset=int(1),                                    # Total number of multi-cell inputs that will be generated per class (or sample and class)
        per_sample = True,                                # For each sample, nsubset samples of ncell are performed
        nfilter_choice= [3,5,7,9], #list(range(3,21)),                 # Range of possible number of filters
        maxpool_percentages=[0.01, 1., 5., 20., 100.],    # list of k-percentage max_pooling
        learning_rate=[0.01], #, 0.1, 1, 10, 50, 100],                               # Learning rate
        max_epochs=50, #50
        patience=5,                                       # Early stopping patience
        nrun=15,  #15                                     # Number of neural network configurations to try (for Hyperparameter optimization)
        regression=False,
        scale=True,                                       # Z-score normalization
        verbose=1
    )

    print(f'Model defined...')
    
    outdir = f'/content/cellcnn_results'  # Results Directory

    print(f'Fitting started...')

    
    datasets = CV_train_datasets
    labels = CV_train_y

In [ ]:
from sklearn.model_selection import StratifiedKFold
def CV_CSNN(dataset, labels, n_cell = 10, n_sub = 3, cv = False, seed = 42, 
                      max_epochs=1, #50
                        patience=5,                                       # Early stopping patience
                        nrun=1):  #15   ):
    

    k = 5
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42) # define class
    cv_results = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(datasets, labels)):
        print(f"Fold {fold + 1}")

        print(f'Creating folds...')
        # Seleziona i dataset e le etichette per train e val
        train_datasets = [datasets[i] for i in train_idx]
        val_datasets   = [datasets[i] for i in val_idx]
        train_labels   = [labels[i] for i in train_idx]
        val_labels     = [labels[i] for i in val_idx]
        print(f'Folds Created!')
        '''
        train_samples_array = df_to_array(train_datasets)
        train_y_array = np.array(train_labels)
    
        val_samples_array = df_to_array(val_datasets)
        val_y_array = np.array(val_labels)
    
    
        model.fit(
            train_samples=train_samples_array,
            train_phenotypes=train_y_array,
            outdir=outdir,
            valid_samples= val_samples_array,
            valid_phenotypes= val_y_array,
            generate_valid_set = False
        )
        '''
        model.fit(
            train_samples=train_datasets,
            train_phenotypes=np.array(train_labels),
            outdir=outdir,
            valid_samples = val_datasets,
            valid_phenotypes=np.array(val_labels),
            generate_valid_set = False
        )

    
        cv_results.append(model.results)
        #hy_pavr_choice.append(model.results['config'])
        #accuracies.append(model.results['accuracies'])
        #best_nets.append(model.results['best_net'])
        #test_pred = model.predict(test_samples_array)
        #predictions.append(test_pred)
    
    
    

    #print(f'Prediction started...')
    #test_pred = model.predict((test_datasets_no_labels))
    print(f'Done')
    return cv_results, model#, model.model_sorted_idx)
    #'''

In [ ]:
%%time
# 1. Pulisci la memoria prima di eseguire
import gc
gc.collect()


seed = 42

model_param, model = CV_CSNN(dataset, labels, n_cell = 10, n_sub = 3, cv = False, seed = seed)

In [ ]:
#model_par, model = model_param
#print(model)

In [ ]:
best_acc = 0
counter = 0

# find the best accuracy
for fold in model_param:
    #print(fold['accuracies'])
    for acc in fold['accuracies']: # for each run performed in the fold
        if acc > best_acc:
            best_acc = acc
            best_fold = counter

    counter += 1
    
print(f'Best accuracy: {best_acc}')
print(f'Fold that performed best: {best_fold}')
# select the cofiguration of the fold that performed best
model.results = model_param[best_fold]

# convert test datasets to numpy array
#test_samples_array = df_to_array(test_dataset)
#test_y_array = np.array(test_y)

# predict using the selected model.results
cv_test_pred = model.predict(test_datasets_no_labels)

pred_phenotypes_array = np.array(phenotype_prediction(cv_test_pred)) # extract predictions

comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy

pred_str = ''

pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
pred_str += f'Trial Accuracy: {accuracy}'
print(pred_str)

In [ ]:
save_path = 'C:\\Users\\admin\\0.Master_Thesis\\'

with open(f'{save_path}cv_results_list.json', "w", encoding="utf-8") as f:
    json.dump(model_param, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{cv_save_path}cv_predictions_list.json', "w", encoding="utf-8") as f:
    json.dump(cv_test_pred, f, default=default_serializer, ensure_ascii=False, indent=2)

In [ ]:
%%time
result_path = '\\results\\'
hy_par_choice = []
predictions = []
predictions_str = []
for i in range(10):


    # extract the length of the smallest dataset in the training dataset
    min_l = min_length(train_dataset)
    
    # define the model
    model = CellCnn(
        ncell=min_l,            #200                        # Number of cells per multi-cell input (sampled from the 'patient' datasets)
        nsubset=int(1),                                    # Total number of multi-cell inputs that will be generated per class (or sample and class)
        per_sample = True,                                # For each sample, nsubset samples of ncell are performed
        nfilter_choice= [3,5,7,9], #list(range(3,21)),                 # Range of possible number of filters
        maxpool_percentages=[0.01, 1., 5., 20., 100.],    # list of k-percentage max_pooling
        learning_rate=[0.01],                               # Learning rate
        max_epochs=50, #50
        patience=5,                                       # Early stopping patience
        nrun=15,  #15                                     # Number of neural network configurations to try (for Hyperparameter optimization)
        regression=False,
        scale=True,                                       # Z-score normalization
        verbose=1
    )

    outdir = '/content/cellcnn_results'  # Results Directory

    # training and validation
    model.fit(
            train_samples=(train_dataset),
            train_phenotypes=np.array(train_y),
            outdir=outdir,
            valid_samples= (val_dataset),
            valid_phenotypes=np.array(val_y),
            generate_valid_set = False
        )
    
    hy_par_choice.append(model.results['config'])

    # prediction
    test_pred = model.predict((test_dataset))
    predictions.append(test_pred)

#save_data
with open(f'{fixed_path}{result_path}hyperparameters_choice_min.json', "w", encoding="utf-8") as f:
    json.dump(hy_par_choice, f, default=default_serializer, ensure_ascii=False, indent=2)

with open(f'{fixed_path}{result_path}predictions_min.json', "w", encoding="utf-8") as f:
    json.dump(predictions, f, default=default_serializer, ensure_ascii=False, indent=2)

#total_trial_parametes = show_hyperparameters(hy_par_choice)


In [ ]:
# load saved data
'''
with open(f'{fixed_path}{result_path}hyperparameters_choice.json', "r", encoding="utf-8") as f:
    hy_par_choice_1 = json.load(f)


with open(f'{fixed_path}{result_path}hyperparameters_choice_min.json', "r", encoding="utf-8") as f:
    hy_par_choice_min = json.load(f)
'''
with open(f'{fixed_path}{result_path}predictions.json', "r", encoding="utf-8") as f:
    predictions_1 = json.load(f)

with open(f'{fixed_path}{result_path}predictions_min.json', "r", encoding="utf-8") as f:
    predictions_min = json.load(f)

In [ ]:
accuracy_list = []
for pred in predictions_1:
    
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

In [ ]:
accuracy_list = []
for pred in predictions_min:
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

In [ ]:
total_trial_parametes = show_hyperparameters(hy_par_choice_1)

accuracy_list = []
for par, pred in zip(total_trial_parametes, predictions_1):
    for line in par:
        print(line)
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')
    

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

In [ ]:
total_trial_parametes = show_hyperparameters(hy_par_choice_min)

accuracy_list = []
for par, pred in zip(total_trial_parametes, predictions_min):
    for line in par:
        print(line)
    pred_phenotypes_array = np.array(phenotype_prediction(pred)) # extract predictions
    comparison = pred_phenotypes_array ==  test_y #checks differencies in prediction
    
    accuracy = np.sum(comparison)/ len(test_y)  #compute accuracy
    accuracy_list.append(accuracy)
    
    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y}\n'
    pred_str += f'Trial Accuracy: {accuracy}'
    print(f'{pred_str}\n')
    

mean_accuracy = np.mean(accuracy_list)
print(f'mean_accuracy over the ten trials: {mean_accuracy}')
accuracy_std = np.std(accuracy_list)
print(f'accuracy_std over the ten trials: {accuracy_std }')

In [ ]:
print(hy_par_choice_1)
for idx, conf in enumerate(hy_par_choice_1):
    print(idx)
    